# Bloomberg pull smoke test (`bdib`)

Replica del Ticker Extract que funciona:
```python
from bbgapi.query import BlpQuery
bquery = BlpQuery().start()
bars = bquery.bdib(
    security="USOSFR10 BGN Curncy",
    event_type="TRADE",
    interval=180,
    start_datetime="2026-06-01T00:00:00",
    end_datetime="2026-09-08T21:00:00",
)
close = bars.set_index("time")["close"]
```

Kernel: el mismo donde ya corre tu extract (con `bbgapi_dependencies()` si hace falta).

In [ ]:
import os, sys
from pathlib import Path
from datetime import datetime, timedelta
import pandas as pd

REPO = Path("/v/region/na/appl/datazone/notebooks2/data/users/arraipau/gh_fwd_screener")
if not REPO.exists():
    REPO = Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("cwd:", Path.cwd())
print("python:", sys.executable)

In [ ]:
# 1) Cliente (igual que tu notebook)
from bbgapi.query import BlpQuery
bquery = BlpQuery().start()
print("started", bquery)

In [ ]:
# 2) Smoke crudo bdib (1 ticker) — debe verse time / close
end = datetime(2026, 9, 8, 21, 0, 0)
start = end - timedelta(days=100)
fmt = "%Y-%m-%dT%H:%M:%S"

bars = bquery.bdib(
    security="USOSFR10 BGN Curncy",
    event_type="TRADE",
    interval=180,
    start_datetime=start.strftime(fmt),
    end_datetime=end.strftime(fmt),
)
print(bars.shape, list(bars.columns))
bars.head()

In [ ]:
# 3) Historia corta vía data.py (mismo bdib, todos los tickers del screener)
from data import _securities, pull_history_into_store, append_live_bar, load_live_store
from curve_config import BBG_INTERVAL_MINUTES, BBG_START_DATE, LIVE_STORE_PATH

print("n_securities", len(_securities()), "interval", BBG_INTERVAL_MINUTES)
print("sample", _securities()[:3], "…", _securities()[-2:])

hist = pull_history_into_store(
    start_date=start.strftime("%Y-%m-%d"),
    end_date=end.strftime("%Y-%m-%d %H:%M:%S"),
    freq=180,
)
print(hist.shape, hist.index[0], "→", hist.index[-1])
hist.iloc[:, :6].tail()

In [ ]:
# 4) Append latest bar + predicción
panel = append_live_bar()
print("store", LIVE_STORE_PATH, "n", len(load_live_store()), "last", panel.index[-1])

from run_prediction import run_once
table = run_once(refresh=False)
table[["model", "structure", "nivel", "y_hat_now", "residual", "y_hat_next"]].head(12)